# Analysez les données de systèmes éducatifs

## Exercice partie 1

### Etape 1 - Chargez les données dans votre Notebook

In [ ]:
import numpy as np
import pandas as pd

# Permet d'afficher toutes les colonnes du dataframe
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
imported_countries = pd.read_csv('./data/EdStatsCountry.csv')
imported_country_series = pd.read_csv('./data/EdStatsCountry-Series.csv')
imported_data = pd.read_csv('./data/EdStatsData.csv')
imported_footnote = pd.read_csv('./data/EdStatsFootNote.csv')
imported_series = pd.read_csv('./data/EdStatsSeries.csv')

### Etape 2 - Collectez les informations basiques sur chaque jeu de données

In [ ]:
# Fonction d'analyse des jeux de données
def analyze(file_name, data, categorial_columns):

    print(f"Le fichier {file_name} contient {data.shape[0]} lignes pour {data.shape[1]} colonnes. Il y à {data.duplicated().sum()} lignes dupliquée.")
    print("\n")

    for column in data.columns:
        print(column)
        nb_cell_empty = data[column].isnull().sum()
        print(f"Colonne {column} : {nb_cell_empty} cellules vides sur {data.shape[0]}, Soit {round((100 * nb_cell_empty) / data.shape[0])}% cellules vides.")

        # On utilise describe pour les colonnes numériques
        if data[column].dtype == 'int64' or data[column].dtype == 'float64':
            print(data[column].describe())

        # On affiche le nombre d'occurence par groupe pour les colonnes catégorielles
        if column in categorial_columns:
            print(data.groupby(column).size())

        print("\n")

#### **Analyse du fichier EdStatsCountry**

Chaque ligne du fichiers contient des informations sur un pays. Par exemple ci-dessus les informations du pays FRANCE


In [ ]:
# Exemple d'une ligne du jeu de donnée
imported_countries.loc[imported_countries['Country Code'] == 'FRA']

In [ ]:
# Supression d'une colonne inconnue
imported_countries.drop('Unnamed: 31', axis=1, inplace=True)

categorial_cols = ['Region', 'Income Group', 'National accounts base year', 'SNA price valuation', 'Lending category', 'Other groups', 'System of National Accounts', 'PPP survey year', 'External debt Reporting status', 'System of trade', 'Government Accounting concept', 'IMF data dissemination standard', 'Latest population census']

# Analyse du fichier
analyze('EdStatsCountry', imported_countries, categorial_cols)

# On renomme cette colonne pour simplifier les jointures + tard
imported_countries.rename(columns={'Country Code' : 'CountryCode'}, inplace=True)

#### **Analyse du fichier EdStatsCountry-Series**

Chaque ligne du fichier contient des informations complémentaires sur les données.

C'est aussi une table de correspondance entre le fichier Pays et le fichier Stats

La clé CountryCode correspond a la clé Country Code dans le fichier EdStatsCountry, et la clé SeriesCode correspond à la clé Indicator Code dans le fichier EdStatsData


In [ ]:
imported_country_series.loc[imported_country_series['CountryCode'] == 'FRA']

In [ ]:
# Supression d'une colonne
imported_country_series.drop('Unnamed: 3', axis=1, inplace=True)

analyze('EdStatsCountrySeries', imported_country_series, [])

#### **Anayse du fichier EdStatsData**

Chaque ligne du fichier contient des statistiques par année

In [ ]:
imported_data.head(2)

In [ ]:
imported_data.drop('Unnamed: 69', axis=1, inplace=True)

analyze('EdStatsData', imported_data, [])

# On renomme la colonne maintenant pour simplifier les jointures plus tard
imported_data.rename(columns={'Indicator Code' : 'SeriesCode', 'Country Code': 'CountryCode'}, inplace=True)

In [ ]:
imported_data.head(2)

#### **Analyse du fichier EdStatsFootNote**

Chaque ligne du fichier contient du détail pour certaines lignes de EdStatsData

La clé SeriesCode correspond à la clé Indicator Code dans le fichier EdStatsData

In [ ]:
imported_footnote.head(2)

In [ ]:
imported_footnote.drop('Unnamed: 4', axis=1, inplace=True)

analyze('EdStatsFootNote', imported_footnote, [])

#### **Analyse du fichier EdStatsSeries**

Chaque ligne du fichier contient des explication concernant le type de donnée enregistré dans chaque cellule par année.

La clé Series Code correspond à la clé Indicator Code dans le fichier EdStatsData

In [ ]:
imported_series.head(2)

In [ ]:
# Supression d'une colonne inconnue
imported_series.drop('Unnamed: 20', axis=1, inplace=True)

analyze('EdStatsSeries', imported_series, [])

imported_series.rename(columns={'Series Code' : 'SeriesCode'}, inplace=True)

### Etape 3 - Réalisez votre premier nettoyage

#### Supression des faux pays

In [ ]:
countries_cleaned = imported_countries[ ~ imported_countries['Region'].isnull()]

Ma méthode naïve a été de téléchargé la liste des country code disponible sur kaggle ici : https://www.kaggle.com/datasets/wbdill/country-codes-iso-3166?resource=download

Et j'ai mergé le fichier EdStatsCountry en inner pour ne garder que les codes des pays qui sont présent dans les deux fichiers.

In [ ]:
import pandas as pd
countries_iso3166b = pd.read_csv('./data/countries_iso3166b.csv', sep=',', na_values=[''], quotechar='"', encoding='ISO-8859-1')
filtered_countries = imported_countries.merge(countries_iso3166b, left_on='CountryCode', right_on='iso3', how='inner')
filtered_countries.head()

Mais j'ai remarqué après ques les faux pays sont ceux du jeu de donnée qui n'ont pas de région associée. On peut filtrer les pays de cette manière.

In [ ]:
countries_cleaned = imported_countries[ ~ imported_countries['Region'].isnull()]

#### Supression des faux pays dans les autres df

##### Méthode 1

En stockant les faux pays dans une liste qui sera utilisée pour le filtrage des différents dataframes.


In [ ]:
# On enregistre les pays a supprimer
print(f'Nombre de pays total : {imported_countries.shape[0]}')
countries_to_remove = imported_countries[imported_countries['Region'].isnull()]
print(f'Country a supprimer : {countries_to_remove.shape[0]}')
print(f'Country restant : {imported_countries.shape[0] - countries_to_remove.shape[0]}')

# On supprime les country series dont le countryCode est dans la liste des pays à supprimer
print(f'Country series avant : {imported_country_series.shape}')
method_one_country_series_filtered = imported_country_series[ ~ imported_country_series['CountryCode'].isin(countries_to_remove['CountryCode'])]
print(f'Country series après : {method_one_country_series_filtered.shape}')

# On supprime les stats data dont le countryCode est dans la liste des pays à supprimer
print(f'Stats data avant : {imported_data.shape}')
method_one_stats_filtered = imported_data[ ~ imported_data['CountryCode'].isin(countries_to_remove['CountryCode'])]
print(f'Stats data après : {method_one_stats_filtered.shape}')

# On supprime les footnote dont le countryCode est dans la liste des pays à supprimer
print(f'Footnote avant : {imported_footnote.shape}')
method_one_footnote_filtered = imported_footnote[ ~ imported_footnote['CountryCode'].isin(countries_to_remove['CountryCode'])]
print(f'Footnote après : {method_one_footnote_filtered.shape}')

##### Méthode 2

En utilisant un inner join entre les pays du dataframe Country nettoyé, et les autres dataframes.

In [ ]:
# On joint le df country au df country_series sur la colonne CountryCode en inner pour s'assurer d'avoir une jointure sans faux pays
print(f'Country series avant : {imported_country_series.shape}')
method_two_country_series_fitered = imported_country_series.merge(countries_cleaned, how='inner')
print(f'Country series avant : {method_two_country_series_fitered.shape}')

# On joint a le df method_two_country_series_fitered au df stats en inner pour s'assurer de ne garder que les stats qui ont un SeriesCode qui est présent dans les SeriesCode de notre df method_two_country_series_fitered
print(f'Stats data avant : {imported_data.shape}')
method_two_stats_filtered = imported_data.merge(method_two_country_series_fitered, how='inner')
print(f'Stats data après : {method_one_stats_filtered.shape}')

# On joint
print(f'Footnote avant : {imported_footnote.shape}')
method_two_footnote_filtered = imported_footnote.merge(method_two_country_series_fitered, how='inner')
print(f'Footnote après : {method_one_footnote_filtered.shape}')

## Exercice partie 2

### Etape 1 - Réduisez le périmètre en utilisant une approche métier

In [ ]:
# Parmi les jeux de données centrés sur les indicateurs, identifiez la colonne qui décrit la catégorie métier à laquelle appartient chaque indicateur.
imported_series['Topic'].value_counts()

Catégories utiles :
- Learning Outcomes (Résultats d'apprentissage)
- Attainment (Niveau d'instruction (ou Niveau d'études atteint)
- Secondary (Secondaire)
- Tertiary (Supérieur)
- Teachers (Enseignants)
- Literacy (Alphabétisation)
- Infrastructure: Communications
- Population (population)

In [ ]:
# Gardez les catégories qui font sens par rapport à la demande de Mark et l’objectif du projet et supprimez les autres.
# significant_topics = ['']
significant_topics = ['Learning Outcomes', 'Attainment', 'Secondary', 'Tertiary', 'Teachers', 'Literacy', 'Infrastructure', 'Population']
significative_series = imported_series[imported_series['Topic'].isin(significant_topics)]

# Calculez le nombre d’indicateurs restants.
print(f'Indicateurs avant filtrage : {imported_series.shape}')
print(f'Indicateurs après filtrage : {significative_series.shape}')

significative_series['Topic'].value_counts()


In [ ]:
significative_series.head()

In [ ]:
# Filtrez l’ensemble des jeux de données pour ne garder que les indicateurs sélectionnés.

print(f'Country series avant : {method_one_country_series_filtered.shape}')
country_series_filtered = method_one_country_series_filtered[method_one_country_series_filtered['SeriesCode'].isin(significative_series['SeriesCode'])]
print(f'Country series après : {country_series_filtered.shape}')

print(f'Stats data avant : {method_one_stats_filtered.shape}')
stats_filtered = method_one_stats_filtered[method_one_stats_filtered['SeriesCode'].isin(significative_series['SeriesCode'])]
print(f'Stats data après : {stats_filtered.shape}')

print(f'Footnote avant : {method_one_footnote_filtered.shape}')
footnote_filtered = method_one_footnote_filtered[method_one_footnote_filtered['SeriesCode'].isin(significative_series['SeriesCode'])]
print(f'Footnote après : {footnote_filtered.shape}')

Dans le fichier Data, interprétez les colonnes représentant les années.

Pourquoi avons-nous des valeurs d’indicateur pour des années futures ?
A but de projections futures

In [ ]:
imported_data.loc[~imported_data['2100'].isnull()]

In [ ]:
# Sur la base de votre compréhension de la problématique métier, filtrez les années en conséquence.
# Je supprime les colonnes des années au delas de 2016

import numpy as np

# On retire les année après 2016 qui sont des projection
columns_to_drop = np.arange(2020,2105,5)
columns_to_drop = [str(col) for col in columns_to_drop]
print(f'Shape avant : {stats_filtered.shape}')
stats_filtered.drop(columns=columns_to_drop, inplace=True)
print(f'Shape après : {stats_filtered.shape}')

# On retire les année avant 2000 qui sont un peu vieille
columns_to_drop = np.arange(1970,2010,1)
columns_to_drop = [str(col) for col in columns_to_drop]
print(f'Shape avant : {stats_filtered.shape}')
stats_filtered.drop(columns=columns_to_drop, inplace=True)
print(f'Shape après : {stats_filtered.shape}')

stats_filtered.head(2)

### Etape 2 - Réduisez le périmètre en utilisant une approche data

In [ ]:
years = np.arange(2010,2018, 1)
cols = [str(col) for col in years]

In [ ]:
# Pour chaque année, calculez la proportion d’indicateurs avec des valeurs renseignées (c’est-à-dire, non manquantes).
stats_by_year = stats_filtered[cols]
stats_by_year.notnull().mean()


In [ ]:
# Pour chaque indicateur, calculez la proportion d’années avec des valeurs renseignées.
stats_by_year = stats_filtered[cols]

stats_filtered['percentage_data_filled'] = stats_by_year.notnull().mean(axis=1)


In [ ]:
# Identifiez les indicateurs particulièrement riches en données en calculant un dataframe contenant par indicateur et par année, le nombre de pays avec une valeur renseignée.

indicator_with_data_for_country = stats_filtered.groupby(['Indicator Name'])[cols].agg('count')
indicator_with_data_for_country


In [ ]:
indicator_with_data_for_country['total_countries'] = indicator_with_data_for_country[cols].mean(axis=1)

In [ ]:
# On ne garde que les indicateurs qui ont au moins 100 pays d dont la valeur est renseignée
indicators_filtered = indicator_with_data_for_country.loc[indicator_with_data_for_country['total_countries'] > 100].sort_values('total_countries', ascending=False)
indicators_filtered

# stats_filtered.loc[stats_filtered['CountryCode'].isin(countries_to_remove['CountryCode'])].shape
#
# stats_filtered.loc[stats_filtered['Indicator Name'] == 'Gross enrolment ratio, secondary, both sexes (%)']

Voici la liste des indicateurs sélectionés qui me semblent pertinent :
- Population of the official age for upper secondary education, both sexes (Lycéens potentiels)
- Population of the official age for tertiary education, both sexes (Étudiants potentiels)
- Population, ages 15-24, total (La tranche d'âge globale cible pour le Lycée et supérieur)
- Enrolment in secondary education, both sexe (Nombre réel d'inscrits)
- Gross enrolment ratio, upper secondary, both sexes (Taux de scolarisation au lycée)
- Gross enrolment ratio, secondary, both sexes (%)

Après dans notre objectif edtech les données sur le topic "Infrastructure" me semble important je vais vérifier la quantité des données qu'on a concernant les indicateurs de cette catégorie-là.

In [ ]:
# Filtrez votre dataframe Data pour ne garder que les indicateurs, pays et années que vous avez jugé pertinents sur la base de vos analyses précédentes.

indicators_selected = [
    'Population of the official age for upper secondary education, both sexes (number)',
    'Population of the official age for tertiary education, both sexes (number)',
    'Enrolment in secondary education, both sexes (number)',
    'Gross enrolment ratio, upper secondary, both sexes (%)',
    'Gross enrolment ratio, secondary, both sexes (%)',
    'Population, age 15, total',
    'Population, age 16, total',
    'Population, age 17, total',
    'Population, age 18, total',
    'Population, age 19, total',
    'Population, age 20, total',
    'Population, age 21, total',
    'Population, age 22, total',
    'Population, age 23, total',
    'Population, age 24, total',
]

final_data = stats_filtered.loc[stats_filtered['Indicator Name'].isin(indicators_selected)]

In [ ]:
final_data.describe()

On remarque que l'année 2017 est totalement vide on la supprime

In [ ]:
# On remarque que l'année 2017 est totalement vide on la supprime, ainsi que percentage_data_filled qui n'est plus utile
final_data.drop(columns=['percentage_data_filled', '2017'], inplace=True)

# On va stocker ces données dans un fichiers csv
final_data.to_csv('./data/selected_datas.csv', index=False)

In [ ]:
data = pd.read_csv('data/selected_datas.csv', sep=',')
data.head(2)

In [ ]:
years = ['2010', '2011', '2012', '2013', '2014', '2015', '2016']

data['nb_empty_cells'] = data[years].isnull().sum(axis=1)

empty_cells_by_country = data.groupby('Country Name').agg({'nb_empty_cells': 'sum'}).sort_values('nb_empty_cells', ascending=False)
empty_cells_by_country = empty_cells_by_country.reset_index()


# On remarque que certains pays on a beaucoup de donnée vide
# On a 15 indicateurs pour 7 années donc 105 données au max par pays
# On supprime tout les pays qui on plus de 50 de valeurs vides car ça risque de fausser les moyenne lors de l'agreggation
countries_to_keep = empty_cells_by_country[empty_cells_by_country['nb_empty_cells'] < (0.5 * 105)]

print(f'Pays avant traitement : {data['Country Name'].unique().shape[0]}')
data_without_empty_countries = data.loc[data['Country Name'].isin(countries_to_keep['Country Name'])]
print(f'Pays avant traitement : {data_without_empty_countries['Country Name'].unique().shape[0]}')

In [ ]:
data_without_empty_countries.drop(columns=['CountryCode', 'SeriesCode'], inplace=True)

In [ ]:
# Agrégez ce dataframe pour en construire un nouveau : chaque ligne doit correspondre à un pays et chaque colonne doit correspondre à un indicateur.
years = ['2010', '2011', '2012', '2013', '2014', '2015', '2016']
data_without_empty_countries['mean_2010_2016'] = data_without_empty_countries[years].mean(axis=1)
data_formated = data_without_empty_countries.pivot_table(index="Country Name", columns="Indicator Name", values='mean_2010_2016', aggfunc='mean')

data_formated.head()

In [ ]:
data_formated.to_csv('./data/data_formated.csv')